In [ ]:
import datetime
import os
import pathlib

REPO_ROOT = pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline")
CUTOFF = datetime.datetime(2026, 1, 1)
CUTOFF_TS = CUTOFF.timestamp()

In [ ]:
bakery_files = list((REPO_ROOT / "cad").glob("*/*/bakery/*"))
print(f"Found {len(bakery_files)} bakery files.")

In [ ]:
recent = []
for f in bakery_files:
    try:
        mtime = os.path.getmtime(f)
    except OSError as e:
        print(f"WARN: could not stat {f}: {e}")
        continue
    if mtime > CUTOFF_TS:
        recent.append((f, mtime))

recent.sort(key=lambda x: x[1])
print(f"{len(recent)} files modified after {CUTOFF.isoformat()}:")

In [ ]:
for f, mtime in recent:
    when = datetime.datetime.fromtimestamp(mtime).isoformat(timespec="seconds")
    print(f"  {when}  {f.relative_to(REPO_ROOT)}")

In [ ]:
targets = {"/".join(f.parts[-4:-2]) for f, mtime in recent}
print(targets)

In [ ]:
processed_files = [f"cad/{target}/processed.max" for target in targets]
bakery_dirs = [f"cad/{target}/bakery" for target in targets]
print("dvc unprotect", *processed_files)
print("dvc add", *processed_files, *bakery_dirs)